# 09 - SKNY Gridding & Tumor Distance Calculation

Grid the spatial transcriptomics data at 10 µm resolution using `stlearn`,
then compute signed Euclidean distance from the tumor boundary for each grid
element using contour detection and graph-based shortest paths.

**Outputs:** `BICRC_SKNY_tumor_{sample}.h5ad` per sample, with `euclidean`
distance in `.obs`.

**Conda environment:** `skny`

## Imports

In [ ]:
import skny as sk

import stlearn as st

import matplotlib.pyplot as plt

import matplotlib.colors as colors

import matplotlib

import pandas as pd

import numpy as np

import cv2

import os

import math

import anndata as ad

import seaborn as sns

import scanpy as sc

import squidpy as sq

import scipy.stats

from PIL import Image

import networkx as nx

## Utility Functions

In [ ]:
class MidpointNormalize(colors.Normalize):

    """Normalise the colorbar so that diverging bars work from a prescribed midpoint.



    Example:

        im = ax.imshow(array, norm=MidpointNormalize(midpoint=0., vmin=-100, vmax=100))

    """



    def __init__(self, vmin=None, vmax=None, midpoint=None, clip=False):

        self.midpoint = midpoint

        colors.Normalize.__init__(self, vmin, vmax, clip)



    def __call__(self, value, clip=None):

        x, y = [self.vmin, self.midpoint, self.vmax], [0, 0.5, 1]

        return np.ma.masked_array(np.interp(value, x, y), np.isnan(value))





def cv2pil(image):

    """Convert an OpenCV image (BGR/BGRA) to a PIL Image (RGB/RGBA)."""

    new_image = image.copy()

    if new_image.ndim == 2:

        pass

    elif new_image.shape[2] == 3:

        new_image = cv2.cvtColor(new_image, cv2.COLOR_BGR2RGB)

    elif new_image.shape[2] == 4:

        new_image = cv2.cvtColor(new_image, cv2.COLOR_BGRA2RGBA)

    new_image = Image.fromarray(new_image)

    return new_image





def calculate_distance(grid, pos_marker_ls=None, neg_marker_ls=None, use_label=None):

    """Compute signed Euclidean distance from the tumor surface for every grid element.



    Parameters

    ----------

    grid : AnnData

        Grid object produced by stlearn.

    pos_marker_ls, neg_marker_ls : list, optional

        Positive / negative marker gene lists (used when *use_label* is None).

    use_label : str, optional

        Column in ``grid.obs`` that flags tumour grids (overrides marker lists).



    Returns

    -------

    AnnData

        The input *grid* with distance information stored in ``grid.shortest``

        and several visualisation arrays in ``grid.uns``.

    """

    N_ROW = len(grid.uns["grid_yedges"]) - 1

    N_COL = len(grid.uns["grid_xedges"]) - 1



    # --- Define positive grid ------------------------------------------------

    if use_label is None:

        pos_series = sum([grid.to_df()[i] for i in pos_marker_ls])

        pos_series = pd.Series(

            [1 if i >= 1 else 0 for i in pos_series], index=pos_series.index

        )

        if neg_marker_ls is not None:

            neg_series = sum([grid.to_df()[i] for i in neg_marker_ls])

            neg_series = pd.Series(

                [1 if i >= 1 else 0 for i in neg_series], index=neg_series.index

            )

            pos_series = pos_series - neg_series

            pos_series = pd.Series(

                [-1 if i == 0 else i for i in pos_series], index=pos_series.index

            )

        else:

            pos_series = pd.Series(

                [-1 if i == 0 else i for i in pos_series], index=pos_series.index

            )



        pos_series.name = "tumor_grid"

        df_grid_tumor = pd.merge(

            pd.DataFrame(index=["grid_" + str(i + 1) for i in range(N_ROW * N_COL)]),

            pos_series,

            right_index=True, left_index=True, how="left",

        ).fillna(-1)

    else:

        pos_series = grid.obs[use_label].copy()

        pos_series.name = "tumor_grid"

        df_grid_tumor = pd.merge(

            pd.DataFrame(index=["grid_" + str(i + 1) for i in range(N_ROW * N_COL)]),

            pos_series,

            right_index=True, left_index=True, how="left",

        ).fillna(-1)



    # --- Generate image of positive grid ------------------------------------

    fig, ax = plt.subplots(figsize=(N_COL, N_ROW), dpi=1, tight_layout=True)

    cmap = matplotlib.cm.viridis

    cmap.set_bad("black", 1.0)

    cmap.set_under(color="black")

    ax.imshow(

        np.array(df_grid_tumor["tumor_grid"]).reshape(N_COL, N_ROW).T,

        cmap=cmap, vmin=0, vmax=1,

    )

    ax.axis("off")

    fig.canvas.draw()

    img = np.array(fig.canvas.renderer.buffer_rgba())

    img = cv2.cvtColor(img, cv2.COLOR_RGBA2BGR)

    plt.close()

    grid.uns["marker"] = img.copy()



    # --- Median + Gaussian filter -------------------------------------------

    img_med = cv2.medianBlur(img, ksize=3)

    img_med = cv2.GaussianBlur(img_med, (3, 3), 0)

    grid.uns["marker_median"] = img_med.copy()



    # --- Contour detection --------------------------------------------------

    threshold = 20

    img_gray = cv2.cvtColor(img_med, cv2.COLOR_BGR2GRAY)

    img_blur = img_gray

    ret, img_binary = cv2.threshold(img_blur, threshold, 255, cv2.THRESH_BINARY)

    grid.uns["marker_binary"] = img_binary.copy()

    contours, hierarchy = cv2.findContours(

        img_binary, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE

    )



    contours_ = []

    for i, s in zip(contours, hierarchy[0]):

        contours_ += [i]



    img_binary = cv2.cvtColor(img_binary, cv2.COLOR_GRAY2BGR)

    img_binary_ = np.where(

        img_binary == (255, 255, 255), (36, 231, 253), (0, 0, 0)

    )

    img_binary_ = np.array(img_binary_, dtype=np.uint8)

    img_color_with_contours = cv2.drawContours(

        img_binary_, contours_, -1, (0, 255, 0), 1

    )

    grid.uns["marker_delineation"] = img_color_with_contours



    # --- Convert image to graph structure -----------------------------------

    image = cv2pil(img_color_with_contours)

    width, height = image.size

    graph = nx.Graph()



    tumor_contour_pixel_ls = []

    tumor_pixel_ls = []

    edge_pixel_ls = []



    for y in range(height):

        for x in range(width):

            pixel_value = image.getpixel((x, y))

            graph.add_node((x, y), color=pixel_value)



            if pixel_value == (0, 255, 0):

                tumor_contour_pixel_ls += [(x, y)]

            elif pixel_value == (253, 231, 36):

                tumor_pixel_ls += [(x, y)]



            if (y == 0) | (y == height - 1) | (x == 0) | (x == width - 1):

                edge_pixel_ls += [(x, y)]



    nodes_ls = list(graph.nodes)



    # Check which nodes are inside contour

    inside_contour_ls = []

    for node in nodes_ls:

        for i in contours_:

            if cv2.pointPolygonTest(i, node, False) == 1:

                inside_contour_ls += [node]

                break



    tumor_pixel_ls = [i for i in tumor_pixel_ls if i in inside_contour_ls]



    # Build edges between adjacent pixels

    for y in range(height):

        for x in range(width):

            current_node = (x, y)

            neighbors = [

                (x, y - 1),  # up

                (x, y + 1),  # down

                (x - 1, y),  # left

                (x + 1, y),  # right

            ]

            for neighbor in neighbors:

                if neighbor in graph.nodes:

                    graph.add_edge(current_node, neighbor)



            # Diagonal edges

            if (y != height) & (x != width):

                node1 = (x, y)

                node2 = (x + 1, y + 1)

                distance = math.sqrt(

                    (node1[0] - node2[0]) ** 2 + (node1[1] - node2[1]) ** 2

                )

                graph.add_edge(node1, node2, weight=distance)



                node1 = (x + 1, y)

                node2 = (x, y + 1)

                distance = math.sqrt(

                    (node1[0] - node2[0]) ** 2 + (node1[1] - node2[1]) ** 2

                )

                graph.add_edge(node1, node2, weight=distance)



    # Remove contour pixels at image edges from the contour set

    tumor_pixel_ls = list(

        set(tumor_pixel_ls)

        | (set(tumor_contour_pixel_ls) & set(edge_pixel_ls))

    )

    tumor_contour_pixel_ls = list(

        set(tumor_contour_pixel_ls)

        - (set(tumor_contour_pixel_ls) & set(edge_pixel_ls))

    )



    # --- Compute shortest paths from tumor surface --------------------------

    shortest_paths = nx.multi_source_dijkstra(

        graph, tumor_contour_pixel_ls, weight="weight"

    )



    df_shotest = pd.DataFrame.from_dict(

        shortest_paths[0], orient="index", columns=["euclidean"]

    )



    # Negative distance for pixels inside the contour (tumor interior)

    df_shotest.loc[

        df_shotest.index[df_shotest.index.isin(tumor_pixel_ls)]

    ] = (

        df_shotest.loc[

            df_shotest.index[df_shotest.index.isin(tumor_pixel_ls)]

        ]

        * -1

    )

    df_shotest["euclidean_round"] = df_shotest["euclidean"].round(0)

    df_nodes = pd.DataFrame(index=nodes_ls)

    df_shotest = pd.merge(

        df_nodes, df_shotest,

        right_index=True, left_index=True, how="left",

    ).fillna(np.nan)



    fig.canvas.draw()

    img = np.array(fig.canvas.renderer.buffer_rgba())

    plt.close()

    grid.uns["shotest"] = img



    # --- Delineation visualisations -----------------------------------------

    col_ls = []

    for i in df_shotest["euclidean_round"]:

        if i == 0:

            col_ls += [(0, 255, 0)]

        elif i < 0:

            col_ls += [(0, 255, 255)]

        else:

            col_ls += [(0, 0, 0)]

    col_arr = np.array(col_ls, dtype=np.uint8).reshape(N_ROW, N_COL, 3)

    grid.uns["marker_median_delineation"] = col_arr.copy()



    # Delineation with 30 um interval

    col_ls = []

    for i in df_shotest["euclidean_round"]:

        if i == 0:

            col_ls += [(0, 255, 0)]

        elif (i % 3 == 0) & (i > 0):

            col_ls += [(0, 0, 255)]

        elif (i % 3 == 0) & (i < 0):

            col_ls += [(0, 150, 255)]

        elif i < 0:

            col_ls += [(0, 255, 255)]

        else:

            col_ls += [(0, 0, 0)]

    col_arr = np.array(col_ls, dtype=np.uint8).reshape(N_ROW, N_COL, 3)

    grid.uns["shotest_30_delineation"] = col_arr.copy()



    # --- Discretisation into regions ----------------------------------------

    df_shotest["region"] = pd.cut(

        df_shotest.dropna()["euclidean"], bins=list(range(-15, 31, 3))

    )



    for i in (

        df_shotest.dropna()

        .region.value_counts()[

            df_shotest.dropna().region.value_counts() > 100

        ]

        .sort_index()

        .index

    ):

        col_ls = []

        for s in df_shotest["region"]:

            if s == i:

                col_ls += [(255, 255, 255)]

            else:

                col_ls += [(0, 0, 0)]

        col_arr = np.array(col_ls, dtype=np.uint8).reshape(N_ROW, N_COL, 3)

        grid.uns[f"shotest_region_{i}_delineation"] = col_arr



    setattr(grid, "shortest", df_shotest)

    return grid

## Configuration

In [ ]:
DATA_DIR = "../data"

RESULTS_DIR = "../results"

## Data Loading

In [ ]:
adata = ad.read_h5ad(os.path.join(DATA_DIR, "bicrc_integrated_annotated_.h5ad"))

adata = adata[

    (adata.obs["Response"] == "SD") & (adata.obs["Timepoint"] == "Resection")

]

adata.uns["spatial"] = adata.obsm["spatial"]

adata

## Per-Sample SKNY Gridding & Distance Calculation

For each sample, grid at 10 µm resolution, compute subcluster dummies, then
calculate signed Euclidean distance from the tumor boundary. Results are saved
as `BICRC_SKNY_tumor_{sample}.h5ad`.

In [ ]:
for sample in adata.obs["Sample"].unique():

    adata_sample = adata[adata.obs["Sample"] == sample]



    # Gridding at 10 um interval

    N_COL = int(

        (adata_sample.obs.imagecol.max() - adata_sample.obs.imagecol.min()) / 10

    )

    N_ROW = int(

        (adata_sample.obs.imagerow.max() - adata_sample.obs.imagerow.min()) / 10

    )

    grid = st.tl.cci.grid(

        adata_sample, n_row=N_ROW, n_col=N_COL, n_cpus=32,

        verbose=False, use_label="subcluster",

    )



    grid.obs = pd.merge(

        grid.obs,

        pd.get_dummies(grid.obs["subcluster"], dtype=int).astype(float).replace(0, -1),

        right_index=True, left_index=True, how="left",

    )



    # Contour tumor region and calculate distance

    grid = calculate_distance(grid, use_label="Tumor")



    df_region = pd.DataFrame(

        np.array(grid.shortest["euclidean"])

        .reshape(N_ROW, N_COL)

        .T.reshape(N_ROW * N_COL),

        index=["grid_" + str(i + 1) for i in range(N_ROW * N_COL)],

        columns=["euclidean"],

    )



    grid.obs = pd.merge(

        grid.obs, df_region,

        right_index=True, left_index=True, how="left",

    )



    cv2pil(grid.uns["marker_delineation"])



    out_path = os.path.join(RESULTS_DIR, f"BICRC_SKNY_tumor_{sample}.h5ad")

    grid.write_h5ad(out_path)

    print(f"Saved {out_path}")